# KKBOX Churn Prediction — CatBoost

기존 3개 Boosting 모델 비교 노트북의 전처리/파생변수 흐름은 유지하되,  
가장 성능이 좋았던 **CatBoost만 사용**하여 학습과 평가를 진행합니다.

## 평가 기준
- **ROC-AUC**: 주 평가 지표
- **PR-AUC**: 불균형 데이터에서 churn 고객 탐지 성능 확인
- Precision / Recall / F1: threshold=0.5 기준 참고
- Accuracy: 참고용

> VS Code에서 실행하고 GitHub에 올리기 쉽게 Colab 전용 코드는 제거했습니다.


## 1. Library & Data Load

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42


ModuleNotFoundError: No module named 'sklearn'

### CatBoost 설치가 안 되어 있을 때만 실행

VS Code의 현재 Jupyter kernel 환경에 CatBoost가 없다면 아래 셀의 주석을 제거하고 한 번 실행하세요.


In [ ]:
# %pip install catboost

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# 내 PC에서 사용하는 실제 데이터 경로
DATA_PATH = Path(
    r"C:\Users\User\OneDrive\Desktop\SKN 부트캠프\2nd_project\SKN36-2nd-1Team\data\processed\integrated_data.csv"
)

# 다른 팀원이 저장소를 clone한 경우를 위한 상대경로 fallback
if not DATA_PATH.exists():
    DATA_PATH = Path("data/processed/integrated_data.csv")

print("Data path:", DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
display(df.head())


## 2. Target 확인

In [ ]:
print("Target count")
display(df["is_churn"].value_counts().sort_index().to_frame())

print("Target ratio")
display(df["is_churn"].value_counts(normalize=True).sort_index().to_frame())


## 3. Split 이전 기본 정리

기존 모델 비교 노트북의 기준을 그대로 사용합니다.

- `bd`: 10~80세만 정상값으로 처리
- 정상 범위를 벗어난 나이는 결측 처리 후 `age_group='unknown'`
- `gender` 결측치는 `Unknown`
- `registration_init_time`은 날짜형으로 변환하되 모델 입력에서는 제외


In [ ]:
df["registration_init_time"] = pd.to_datetime(
    df["registration_init_time"],
    format="%Y%m%d",
    errors="coerce"
)

df.loc[~df["bd"].between(10, 80), "bd"] = np.nan

df["age_group"] = pd.cut(
    df["bd"],
    bins=[9, 19, 29, 39, 49, 59, 69, 80],
    labels=["10s", "20s", "30s", "40s", "50s", "60s", "70_80"]
)

df["age_group"] = (
    df["age_group"]
    .cat.add_categories("unknown")
    .fillna("unknown")
)

df["gender"] = df["gender"].fillna("Unknown")


## 4. Train / Test Split

In [ ]:
X = df.drop(columns="is_churn")
y = df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print(f"Train churn ratio: {y_train.mean():.4f}")
print(f"Test churn ratio : {y_test.mean():.4f}")


## 5. 결측치 처리 및 불필요 변수 제거

Train 데이터에서 계산한 값만 이용해 Test 데이터를 처리하여 데이터 누수를 방지합니다.

- `city`: 제외
- `registered_via`: Train 최빈값
- `last_auto_renew`, `last_is_cancel`, `last_plan_days`: Train 최빈값
- `days_since_last_log`: Train 평균
- `registration_init_time`, `cancel_on_last_date`, `days_to_expire`, `bd`: 모델 입력에서 제외


In [ ]:
X_train = X_train.drop(columns=["city"])
X_test = X_test.drop(columns=["city"])

registered_via_mode = X_train["registered_via"].mode()[0]
X_train["registered_via"] = X_train["registered_via"].fillna(registered_via_mode)
X_test["registered_via"] = X_test["registered_via"].fillna(registered_via_mode)

for col in ["last_auto_renew", "last_is_cancel", "last_plan_days"]:
    fill_value = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)

last_log_mean = X_train["days_since_last_log"].mean()
X_train["days_since_last_log"] = X_train["days_since_last_log"].fillna(last_log_mean)
X_test["days_since_last_log"] = X_test["days_since_last_log"].fillna(last_log_mean)

drop_cols = [
    "registration_init_time",
    "cancel_on_last_date",
    "days_to_expire",
    "bd"
]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

print("Remaining nulls before feature engineering")
display(
    pd.DataFrame({
        "train_null": X_train.isna().sum(),
        "test_null": X_test.isna().sum()
    }).query("train_null > 0 or test_null > 0")
)


## 6. Feature Engineering

기존 비교 노트북에서 사용한 파생변수를 동일하게 생성합니다.

분모가 0일 때는 `NaN`으로 두어 `inf` 발생을 막습니다.  
CatBoost는 수치형 `NaN`을 자체적으로 처리할 수 있습니다.


In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


def add_features(data):
    data = data.copy()

    # 0 값 자체가 의미를 가질 수 있는 보조 flag
    data["no_unique_song"] = (data["total_num_unq"] == 0).astype(int)
    data["zero_plan_days"] = (data["last_plan_days"] == 0).astype(int)

    # 거래 관련
    data["cancel_rate"] = safe_divide(
        data["cancel_count"],
        data["transaction_count"]
    )

    data["avg_payment"] = safe_divide(
        data["total_payment"],
        data["transaction_count"]
    )

    # 이용 로그 관련
    data["avg_daily_secs"] = safe_divide(
        data["total_secs"],
        data["activity_days"]
    )

    data["avg_daily_complete"] = safe_divide(
        data["total_num_100"],
        data["activity_days"]
    )

    data["avg_daily_unq"] = safe_divide(
        data["total_num_unq"],
        data["activity_days"]
    )

    data["complete_per_unq"] = safe_divide(
        data["total_num_100"],
        data["total_num_unq"]
    )

    # 결제 효율 관련
    data["payment_per_plan_day"] = safe_divide(
        data["avg_payment"],
        data["last_plan_days"]
    )

    return data


X_train = add_features(X_train)
X_test = add_features(X_test)


In [ ]:
new_features = [
    "cancel_rate",
    "avg_payment",
    "avg_daily_secs",
    "avg_daily_complete",
    "avg_daily_unq",
    "complete_per_unq",
    "payment_per_plan_day"
]

display(X_train[new_features].describe().T)

print("NaN generated by zero denominators")
display(
    pd.DataFrame({
        "train_null": X_train[new_features].isna().sum(),
        "test_null": X_test[new_features].isna().sum()
    })
)

numeric_cols = X_train.select_dtypes(include=np.number).columns

print("Train inf count:", np.isinf(X_train[numeric_cols]).sum().sum())
print("Test inf count :", np.isinf(X_test[numeric_cols]).sum().sum())


## 7. CatBoost 입력 데이터 준비

- `msno`: 고객 식별자이므로 모델 입력에서 제외
- 범주형 변수: `gender`, `registered_via`, `age_group`
- CatBoost는 범주형 변수를 One-Hot Encoding 없이 직접 처리


In [ ]:
X_train_model = X_train.drop(columns=["msno"]).copy()
X_test_model = X_test.drop(columns=["msno"]).copy()

cat_features = [
    "gender",
    "registered_via",
    "age_group"
]

# CatBoost categorical feature는 문자열로 통일
for col in cat_features:
    X_train_model[col] = X_train_model[col].astype(str)
    X_test_model[col] = X_test_model[col].astype(str)

print("Train shape:", X_train_model.shape)
print("Test shape :", X_test_model.shape)
print("Categorical features:", cat_features)


## 8. Class Imbalance Weight

Churn 고객 비율이 낮으므로 Train 데이터의 실제 class ratio를 이용해 positive class weight를 계산합니다.

SMOTE는 사용하지 않습니다.


In [ ]:
n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()

scale_pos_weight = n_negative / n_positive

print("Negative:", n_negative)
print("Positive:", n_positive)
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


## 9. CatBoost 학습

In [ ]:
cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    custom_metric=["PRAUC"],
    class_weights=[1.0, scale_pos_weight],
    random_seed=RANDOM_STATE,
    verbose=100
)

cat_model.fit(
    X_train_model,
    y_train,
    cat_features=cat_features
)


## 10. 모델 성능 평가

In [ ]:
pred_proba = cat_model.predict_proba(X_test_model)[:, 1]
pred = (pred_proba >= 0.5).astype(int)

metrics = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1",
        "Accuracy"
    ],
    "Score": [
        roc_auc_score(y_test, pred_proba),
        average_precision_score(y_test, pred_proba),
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        f1_score(y_test, pred, zero_division=0),
        accuracy_score(y_test, pred)
    ]
})

display(metrics.style.format({"Score": "{:.4f}"}))


In [ ]:
print(classification_report(y_test, pred))

## 11. Normalized Confusion Matrix

In [ ]:
cm_norm = confusion_matrix(
    y_test,
    pred,
    normalize="true"
) * 100

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".1f",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CatBoost — Normalized Confusion Matrix (%)")
plt.show()


## 12. Feature Importance

In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train_model.columns,
        "importance": cat_model.get_feature_importance()
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

plt.figure(figsize=(8, 7))
sns.barplot(
    data=feature_importance.head(20),
    x="importance",
    y="feature"
)
plt.title("CatBoost — Top 20 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
